## Fase 3: Reconocimiento de Entidades Nombradas (NER)

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">3.1. Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el porcentaje de comentarios que contienen entidades NER en cada grupo?</span>

In [28]:
# Inicializamos contadores
comentarios_sin_odio = 0
comentarios_con_odio = 0
comentarios_sin_odio_ner = 0
comentarios_con_odio_ner = 0

# Número de filas
num_filas = sub_data.shape[0]

# Bucle principal
for i in range(num_filas):

    # Solo analizamos comentarios
    if sub_data.loc[i, "TIPO DE MENSAJE"] != "COMENTARIO":
        continue

    texto = str(sub_data.loc[i, "CONTENIDO A ANALIZAR"])
    doc = nlp(texto)

    tiene_ner = len(doc.ents) > 0

    # Comentarios sin odio
    if sub_data.loc[i, "INTENSIDAD"] == 0:
        comentarios_sin_odio += 1
        if tiene_ner:
            comentarios_sin_odio_ner += 1

    # Comentarios con odio
    elif sub_data.loc[i, "INTENSIDAD"] > 0:
        comentarios_con_odio += 1
        if tiene_ner:
            comentarios_con_odio_ner += 1

# Cálculo de porcentajes
porc_sin_odio = (comentarios_sin_odio_ner / comentarios_sin_odio) * 100
porc_con_odio = (comentarios_con_odio_ner / comentarios_con_odio) * 100

print(f"Porcentaje de comentarios sin odio con entidades NER: {porc_sin_odio:.2f}%")
print(f"Porcentaje de comentarios con odio con entidades NER: {porc_con_odio:.2f}%")


Porcentaje de comentarios sin odio con entidades NER: 42.88%
Porcentaje de comentarios con odio con entidades NER: 33.96%


<hr>
El reconocimiento de entidades nombradas (NER) es una técnica de procesamiento del lenguaje natural que identifica y clasifica automáticamente menciones a entidades del mundo real dentro del texto (persona, lugar, organización...).

En nuestro caso, el 42.88 % de los comentarios sin odio presenta al menos una entidad NER y el 33.96 % de los comentarios con odio presenta al menos una entidad NER, es decir, los comentarios sin odio presentan entidades NER con mayor frecuencia.

Podemos interpretar estos resultados concluyendo:
- El discurso no odioso tiende más a hablar de personas concretas y mencionar organizaciones, lugares, hechos...
- El discurso odioso es más genérico, usa más insultos, adjetivos, y no necesita referirse a entidades nombradas.

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">3.2. Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el porcentaje de comentarios que contienen entidades NER de tipo PERSON, ORG, LOC y NORP en cada grupo?</span>

In [29]:
# Inicializamos contadores
conteo_ner_sin_odio = {
    "PER": 0,
    "LOC": 0,
    "ORG": 0,
    "NORP": 0,
    "Total": 0 # OJO: el total no es de este diccionario, es la suma total de todas la entidades que se cuentan (pueden no ser estas cuatro)
}

conteo_ner_con_odio = {
    "PER": 0,
    "LOC": 0,
    "ORG": 0,
    "NORP": 0,
    "Total": 0 # OJO: el total no es de este diccionario, es la suma total de todas la entidades que se cuentan (pueden no ser estas cuatro)
}

# Número de filas
num_filas = sub_data.shape[0]

# Bucle principal
for i in range(num_filas):

    # Solo analizamos comentarios
    if sub_data.loc[i, "TIPO DE MENSAJE"] != "COMENTARIO":
        continue

    texto = str(sub_data.loc[i, "CONTENIDO A ANALIZAR"])
    doc = nlp(texto)

    intensidad = sub_data.loc[i, "INTENSIDAD"]

    if intensidad == 0:
        conteo = conteo_ner_sin_odio
    elif intensidad > 0:
        conteo = conteo_ner_con_odio
    else:
        continue

    for ent in doc.ents:
        conteo["Total"] += 1
        
        if ent.label_ in conteo:
            conteo[ent.label_] += 1

# Resultados
print("Las proporciones de entidades NER en comentarios sin odio son:")
for clave, valor in conteo_ner_sin_odio.items():
    if clave != "Total":
        print(f"  {clave}: {valor} ({valor/conteo_ner_sin_odio['Total']*100:.2f}%)")
print(f"  Total: {conteo_ner_sin_odio['Total']}")

print("Las proporciones de entidades NER en comentarios con odio son:")
for clave, valor in conteo_ner_con_odio.items():
    if clave != "Total":
        print(f"  {clave}: {valor} ({valor/conteo_ner_con_odio['Total']*100:.2f}%)")
print(f"  Total: {conteo_ner_con_odio['Total']}")

Las proporciones de entidades NER en comentarios sin odio son:
  PER: 9739 (32.99%)
  LOC: 12783 (43.30%)
  ORG: 3956 (13.40%)
  NORP: 0 (0.00%)
  Total: 29523
Las proporciones de entidades NER en comentarios con odio son:
  PER: 143 (43.60%)
  LOC: 88 (26.83%)
  ORG: 50 (15.24%)
  NORP: 0 (0.00%)
  Total: 328


<hr>
En los comentarios sin odio predominan ligeramente las entidades de localización frente a las de persona, lo que sugiere un discurso más contextualizado. En cambio, en los comentarios con odio se observa una mayor proporción relativa de entidades de persona respecto a las de localización. En consecuencia, podemos inferir que, cuando se emplean entidades nombradas, los comentarios odiosos tienden a dirigirse a individuos concretos.